# BÀI TẬP 2: TÌM TẬP PHỔ BIẾN CHO ĐỀ XUẤT PHIM
Mục tiêu: Tìm các nhóm phim thường xuyên được người dùng yêu thích cùng nhau (Tập phổ biến) từ dữ liệu đánh giá phim MovieLens 100K.

## 1. Import thư viện và Nạp dữ liệu
Dữ liệu `u.data` của MovieLens là file văn bản phân tách bằng tab (`\t`), không có header.

In [4]:
import pandas as pd
import sys

# Đọc file dữ liệu u.data
# Lưu ý: Đảm bảo file 'u.data' nằm cùng thư mục với file code này
data_filename = "data\ml-100k" + "/u.data"

all_ratings = pd.read_csv(data_filename, delimiter="\t", header=None, 
                          names=["UserID", "MovieID", "Rating", "Datetime"])

# Chuyển đổi định dạng thời gian (tùy chọn, để hiển thị cho đẹp)
all_ratings["Datetime"] = pd.to_datetime(all_ratings['Datetime'], unit='s')

# Hiển thị 5 dòng đầu
print("Dữ liệu gốc:")
print(all_ratings.head())

Dữ liệu gốc:
   UserID  MovieID  Rating            Datetime
0     196      242       3 1997-12-04 15:55:49
1     186      302       3 1998-04-04 19:22:22
2      22      377       1 1997-11-07 07:18:36
3     244       51       2 1997-11-27 05:02:03
4     166      346       1 1998-02-02 05:33:16


<>:6: SyntaxWarning: invalid escape sequence '\m'
<>:6: SyntaxWarning: invalid escape sequence '\m'
C:\Users\chinh\AppData\Local\Temp\ipykernel_7684\379626367.py:6: SyntaxWarning: invalid escape sequence '\m'
  data_filename = "data\ml-100k" + "/u.data"


## 2. Tiền xử lý dữ liệu
Chúng ta định nghĩa một bộ phim được coi là "Yêu thích" (Favorable) nếu người dùng đánh giá lớn hơn 3 sao. Để thuật toán chạy nhanh trong bài thực hành (demo), chúng ta sẽ chỉ lấy dữ liệu của 200 người dùng đầu tiên.

In [5]:
# Tạo cột mới 'Favorable' (True nếu Rating > 3)
all_ratings["Favorable"] = all_ratings["Rating"] > 3

# Lọc lấy dữ liệu của 200 User đầu tiên để giảm tải tính toán cho bài tập mẫu
ratings = all_ratings[all_ratings['UserID'].isin(range(200))]

# Chỉ lấy những dòng có đánh giá là "Thích" (Favorable)
favorable_ratings = ratings[ratings["Favorable"]]

print("Kích thước dữ liệu sau khi lọc (200 User đầu & Rating > 3):", favorable_ratings.shape)

Kích thước dữ liệu sau khi lọc (200 User đầu & Rating > 3): (11043, 5)


## 3. Tạo cấu trúc dữ liệu giao dịch
Gom nhóm các phim mà mỗi người dùng đã "Like" lại thành một tập hợp (set).

In [6]:
# Tạo dictionary: Key là UserID, Value là tập hợp (frozenset) các MovieID mà user đó thích
favorable_reviews_by_users = dict((k, frozenset(v.values)) 
                                  for k, v in favorable_ratings.groupby("UserID")["MovieID"])

# Tạo DataFrame đếm số lượng like cho từng phim để chuẩn bị cho bước L1
num_favorable_by_movie = ratings[["MovieID", "Favorable"]].groupby("MovieID").sum()

# Xem thử 5 phim được like nhiều nhất trong tập mẫu
print("\nTop 5 phim được yêu thích nhất (trong tập mẫu):")
print(num_favorable_by_movie.sort_values(by="Favorable", ascending=False).head())


Top 5 phim được yêu thích nhất (trong tập mẫu):
         Favorable
MovieID           
50             100
100             89
258             83
181             79
174             74


## 4. Định nghĩa hàm tìm tập phổ biến (Apriori Step)
Đây là hàm cốt lõi để sinh ra các tập ứng viên `k+1` từ các tập phổ biến `k`.

In [7]:
from collections import defaultdict

def find_frequent_itemsets(favorable_reviews_by_users, k_1_itemsets, min_support):
    """
    Hàm tìm tập phổ biến kích thước k dựa trên tập phổ biến kích thước k-1
    """
    counts = defaultdict(int)
    
    # Duyệt qua từng user và danh sách phim họ thích
    for user, reviews in favorable_reviews_by_users.items():
        
        # Duyệt qua các tập phổ biến đã tìm được ở bước trước (k-1)
        for itemset in k_1_itemsets:
            
            # Nếu tập k-1 này là tập con của các phim user đã thích
            if itemset.issubset(reviews):
                
                # Tìm các phim khác mà user cũng thích nhưng chưa có trong itemset
                for other_reviewed_movie in reviews - itemset:
                    
                    # Tạo tập mới (k phần tử) bằng cách gộp phim mới vào
                    current_superset = itemset | frozenset((other_reviewed_movie,))
                    counts[current_superset] += 1
    
    # Chỉ trả về các tập có số lần xuất hiện >= min_support
    return dict([(itemset, frequency) for itemset, frequency in counts.items() if frequency >= min_support])

## 5. Thực thi thuật toán Apriori
Bắt đầu từ tập phổ biến 1 phần tử (L1), sau đó dùng vòng lặp để tìm L2, L3... cho đến khi không tìm thấy tập nào nữa.

In [8]:
# Khởi tạo dictionary lưu trữ tất cả các tập phổ biến theo độ dài
frequent_itemsets = {} 

# Thiết lập ngưỡng hỗ trợ tối thiểu (min_support)
# Tức là phải có ít nhất 50 người cùng thích nhóm phim này mới được coi là phổ biến
min_support = 50

# --- BƯỚC 1: Tìm tập phổ biến cấp 1 (L1) ---
# k=1 candidates are the movies with more than min_support favorable reviews
frequent_itemsets[1] = dict((frozenset((movie_id,)), row["Favorable"])
                            for movie_id, row in num_favorable_by_movie.iterrows()
                            if row["Favorable"] > min_support)

print("Có {} phim với hơn {} lượt thích.".format(len(frequent_itemsets[1]), min_support))

# --- BƯỚC 2: Vòng lặp tìm L2, L3, ... ---
# Duyệt k từ 2 đến 20
for k in range(2, 20):
    # Sinh tập ứng viên k từ tập phổ biến k-1
    cur_frequent_itemsets = find_frequent_itemsets(favorable_reviews_by_users, 
                                                   frequent_itemsets[k-1], 
                                                   min_support)
    
    if len(cur_frequent_itemsets) == 0:
        print("Không tìm thấy tập phổ biến nào có độ dài {}".format(k))
        sys.stdout.flush()
        break
    else:
        print("Tôi tìm thấy {} tập phổ biến có độ dài {}".format(len(cur_frequent_itemsets), k))
        sys.stdout.flush()
        
        # Lưu lại kết quả
        frequent_itemsets[k] = cur_frequent_itemsets

# Xóa tập phổ biến độ dài 1 vì chúng ta thường quan tâm đến sự kết hợp (từ 2 phim trở lên)
del frequent_itemsets[1]

Có 16 phim với hơn 50 lượt thích.
Tôi tìm thấy 93 tập phổ biến có độ dài 2
Tôi tìm thấy 295 tập phổ biến có độ dài 3
Tôi tìm thấy 593 tập phổ biến có độ dài 4
Tôi tìm thấy 785 tập phổ biến có độ dài 5
Tôi tìm thấy 677 tập phổ biến có độ dài 6
Tôi tìm thấy 373 tập phổ biến có độ dài 7
Tôi tìm thấy 126 tập phổ biến có độ dài 8
Tôi tìm thấy 24 tập phổ biến có độ dài 9
Tôi tìm thấy 2 tập phổ biến có độ dài 10
Không tìm thấy tập phổ biến nào có độ dài 11


## 6. Kết quả
Biến `frequent_itemsets` giờ đây chứa các nhóm phim (được biểu diễn bằng ID) thường xuyên được xem cùng nhau.

In [9]:
print("\nKẾT QUẢ MẪU")
# In thử một vài tập phổ biến độ dài 2
if 2 in frequent_itemsets:
    print("Một số cặp phim phổ biến (MovieID):")
    count = 0
    for itemset, support in frequent_itemsets[2].items():
        print(f"Items: {list(itemset)} - Support: {support}")
        count += 1
        if count >= 5: break


KẾT QUẢ MẪU
Một số cặp phim phổ biến (MovieID):
Items: [1, np.int64(7)] - Support: 62
Items: [1, np.int64(50)] - Support: 100
Items: [np.int64(56), 1] - Support: 64
Items: [np.int64(64), 1] - Support: 60
Items: [1, np.int64(79)] - Support: 62
